In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("car_fuel_efficiency_2026.csv")

In [2]:
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


In [3]:
df.isnull().sum()

model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64

In [4]:
df.select_dtypes(include=['number']).columns.tolist()

['model_year',
 'num_doors',
 'engine_displacement',
 'num_cylinders',
 'horsepower',
 'vehicle_weight',
 'acceleration',
 'fuel_efficiency_mpg']

In [5]:
df['horsepower'] = df['horsepower'].fillna(0)

df['acceleration'] = df['acceleration'].fillna(0)

In [6]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)

df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=1)

In [7]:
y_train = df_train.pop('fuel_efficiency_mpg')

In [8]:
from sklearn.feature_extraction import DictVectorizer

train_dicts = df_train.to_dict(orient='records')

dv = DictVectorizer(sparse=True)

x_train = dv.fit_transform(train_dicts)

In [9]:
from sklearn.tree import DecisionTreeRegressor

dt_model = DecisionTreeRegressor(max_depth=1)

dt_model.fit(x_train,y_train)

,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.For an example of how ``max_depth`` influences the model, see:ref:`sphx_glr_auto_examples_tree_plot_tree_regression.py`.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 0.24 Poisson deviance criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split among considered features for this split.",'best'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`.Note: splitting may inspect more than ``max_features`` features ifneeded to find a valid split.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with 

In [10]:
from sklearn.tree import export_text
print(export_text(dt_model,feature_names=list(dv.get_feature_names_out())))

|--- model_year <= 1997.50
|   |--- value: [28.61]
|--- model_year >  1997.50
|   |--- value: [31.17]



In [11]:
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(n_estimators=10,
                                 random_state=1,
                                 n_jobs=-1)

rf_model.fit(x_train,y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",10
,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",-1
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",1
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max

In [12]:
y_val = df_val.pop('fuel_efficiency_mpg')

In [13]:
from sklearn.metrics import root_mean_squared_error

val_dicts = df_val.to_dict(orient='records')

x_val = dv.transform(val_dicts)

y_rf_pred = rf_model.predict(x_val)

rmse_rf = root_mean_squared_error(y_val,y_rf_pred)

print(f"RMSE for Random forest is {rmse_rf:.3f}")

RMSE for Random forest is 1.833


In [14]:
rmse_scores = []

for est in [10, 50, 100, 150]:
    rf_model_est = RandomForestRegressor(n_estimators=est,
                                         random_state=1,
                                         n_jobs=-1)
    rf_model_est.fit(x_train,y_train)

    y_rf_est_pred = rf_model_est.predict(x_val)

    rmse_est = root_mean_squared_error(y_val,y_rf_est_pred)

    rmse_scores.append((est,round(rmse_est,3)))

In [15]:
rmse_scores

[(10, 1.833), (50, 1.774), (100, 1.77), (150, 1.769)]

In [16]:
rf_dep_est_scores = []
est_mean_scores = []

for dep in [10, 15, 20, 25]:
    for est in [10, 50, 100, 150]:
        rf_model_est_dep = RandomForestRegressor(max_depth=dep,
                                                 n_estimators=est,
                                                 random_state=1,
                                                 n_jobs=-1)
        rf_model_est_dep.fit(x_train,y_train)
    
        y_rf_est_dep_pred = rf_model_est_dep.predict(x_val)
    
        rmse_est_dep = root_mean_squared_error(y_val,y_rf_est_dep_pred)

        est_mean_scores.append(rmse_est_dep)
        
    rf_dep_est_scores.append((dep,round(np.mean(est_mean_scores),3)))
    est_mean_scores.clear()

In [17]:
rf_dep_est_scores

[(10, np.float64(1.754)),
 (15, np.float64(1.784)),
 (20, np.float64(1.787)),
 (25, np.float64(1.786))]

In [20]:
feat_imp_model = RandomForestRegressor(max_depth=20,
                                       n_estimators=10,
                                       random_state=1,
                                       n_jobs=-1)

feat_imp_model.fit(x_train,y_train)

importance = list(zip(feat_imp_model.feature_importances_,dv.get_feature_names_out()))

[(np.float64(0.054997632153652054), 'acceleration'),
 (np.float64(0.007950634892065116), 'drivetrain=All-wheel drive'),
 (np.float64(0.020062226629861492), 'drivetrain=Front-wheel drive'),
 (np.float64(0.005638475273929295), 'drivetrain=Rear-wheel drive'),
 (np.float64(0.06128144481858751), 'engine_displacement'),
 (np.float64(0.016570533096713348), 'fuel_type=Diesel'),
 (np.float64(0.16071245288752403), 'fuel_type=Gasoline'),
 (np.float64(0.016271934329480736), 'fuel_type=Hybrid'),
 (np.float64(0.07827124720421207), 'horsepower'),
 (np.float64(0.2961071534403154), 'model_year'),
 (np.float64(0.0073182361352048), 'num_cylinders'),
 (np.float64(0.0194014677039506), 'num_doors'),
 (np.float64(0.006514947101173604), 'origin=Asia'),
 (np.float64(0.0034364358107329975), 'origin=Europe'),
 (np.float64(0.042151742775477855), 'origin=USA'),
 (np.float64(0.20331343574711905), 'vehicle_weight')]

In [22]:
importance.sort(reverse=True)

In [23]:
importance

[(np.float64(0.2961071534403154), 'model_year'),
 (np.float64(0.20331343574711905), 'vehicle_weight'),
 (np.float64(0.16071245288752403), 'fuel_type=Gasoline'),
 (np.float64(0.07827124720421207), 'horsepower'),
 (np.float64(0.06128144481858751), 'engine_displacement'),
 (np.float64(0.054997632153652054), 'acceleration'),
 (np.float64(0.042151742775477855), 'origin=USA'),
 (np.float64(0.020062226629861492), 'drivetrain=Front-wheel drive'),
 (np.float64(0.0194014677039506), 'num_doors'),
 (np.float64(0.016570533096713348), 'fuel_type=Diesel'),
 (np.float64(0.016271934329480736), 'fuel_type=Hybrid'),
 (np.float64(0.007950634892065116), 'drivetrain=All-wheel drive'),
 (np.float64(0.0073182361352048), 'num_cylinders'),
 (np.float64(0.006514947101173604), 'origin=Asia'),
 (np.float64(0.005638475273929295), 'drivetrain=Rear-wheel drive'),
 (np.float64(0.0034364358107329975), 'origin=Europe')]

In [25]:
!pip install xgboost



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.6/57.6 MB 36.0 MB/s  0:00:01s eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 305.1/305.1 MB 22.0 MB/s  0:00:13 eta 0:00:010:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]━━━ 1/2 [xgboost]

[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [33]:
import xgboost as xgb

d_train = xgb.DMatrix(x_train, label=y_train, feature_names=list(dv.get_feature_names_out()))
d_val = xgb.DMatrix(x_val, label=y_val, feature_names=list(dv.get_feature_names_out()))

watchlist = [(d_train, 'train'), (d_val, 'val')]

xgb_params_03 = {'eta': 0.3, 
                 'max_depth': 6,
                 'min_child_weight': 1,
                
                 'objective': 'reg:squarederror',
                 'nthread': 8,
                
                 'seed': 1,
                 'verbosity': 1
}

xgb_03_result = {}

xgb.train(xgb_params_03,
          d_train,
          num_boost_round = 100,
          evals = watchlist,
          evals_result = xgb_03_result,
          verbose_eval = False)

xgb_01_result = {}

xgb_01_param = {'eta':0.1,
                'max_depth':6,
                'min_child_weight':1,

                'objective':'reg:squarederror',
                'nthread':8,

                'seed':1,
                'verbosity':1
               }

xgb.train(xgb_01_param,
          d_train,
          num_boost_round = 100,
          evals = watchlist,
          evals_result = xgb_01_result,
          verbose_eval = False)

print(f"RMSE for 0.3 eta xgboost is {round(xgb_03_result['val']['rmse'][-1],3)}")
print(f"RMSE for 0.1 eta xgboost is {round(xgb_01_result['val']['rmse'][-1],3)}")

RMSE for 0.3 eta xgboost is 1.826
RMSE for 0.1 eta xgboost is 1.725
